In [3]:
import ollama 
from scraper import fetch_website_links, fetch_website_contents
import json
from IPython.display import display, update_display,Markdown

In [4]:
def rerank_relevant_links(url):
    system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)

    response = ollama.chat(
        model = "qwen2.5:7b",
        messages = [{"role":"system","content":system_prompt},{"role":"user","content":user_prompt},{"role":"assistant","content":"{"}]
    )
    response = "{" + response.message.content
    relevant_links = json.loads(response)
    return relevant_links
    

In [21]:
relevant_links = rerank_relevant_links("https://www.apple.com")
relevant_links

{'links': [{'type': 'about page', 'url': 'https://www.apple.com'},
  {'type': 'careers page', 'url': 'https://www.apple.com/careers/us/'}]}

In [5]:
def aggregate_websites_contents(url):
    links = rerank_relevant_links(url)
    main_content = fetch_website_contents(url)
    main_content = f"##Landing Page:\n\n {main_content}\n\n##Relevant Links:\n\n"
    for link in links["links"]:
        main_content += f"\n\n ###Link: {link["type"]}\n\n"
        main_content += fetch_website_contents(link["url"])
    return main_content

In [26]:
content = aggregate_websites_contents("https://www.apple.com")
content

'##Landing Page:\n\n Apple\n\nApple\nApple\nStore\nMac\niPad\niPhone\nWatch\nVision\nAirPods\nTV & Home\nEntertainment\nAccessories\nSupport\n0\n+\nSurprise and shine.\nWatch a special Apple\xa0Event online on 9/9 at 10 a.m. PT.\nAdd to calendar\nCollege, sorted.\nGet a gift card from $100 to $150\n*\nwhen you buy Mac or iPad with education savings.\nShop\niPhone\nMeet the latest iPhone lineup.\nLearn more\nShop iPhone\nMac mini\nNow with M6 and M5 Pro.\nAvailable starting 9.22\nLearn more\nPre-order\nMacBook\xa0Air\nNow supercharged by M5.\nLearn more\nBuy\niPad Air\nNow supercharged by M4.\nLearn more\nBuy\niPad\xa0Pro\nAdvanced AI performance and\xa0game-changing capabilities.\nLearn more\nBuy\nApple Trade In\nGet credit toward your next iPhone when you trade in an eligible device.\n1\nGet your estimate\nApple Card\nGet up to 3% Daily\xa0Cash back with every purchase.\nLearn more\nApply now\nEndless entertainment.\nItem 1\nItem 2\nItem 3\nItem 4\nItem 5\nItem 6\nItem 7\nItem 8\nItem

In [6]:
def create_brochure(company_name:str,url:str):
    system_prompt = """
    You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    website_content = aggregate_websites_contents(url)
    user_prompt += "\n".join(website_content)
    user_prompt = user_prompt[:5_000]

    response = ollama.chat(
        model = "qwen2.5:7b",
        messages = [{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}]
    )
    return response.message.content
    

In [40]:
brochure = create_brochure("apple","https://www.apple.com")
display(Markdown(brochure))

# Apple: Innovate and Inspire

## Welcome to Apple

Apple is more than just a tech company; it's a community of innovators and visionaries. Founded in 1976, Apple has grown from a small garage startup into a global leader in technology and design. Known for its sleek, user-friendly products and cutting-edge innovation, Apple continues to shape the future of technology.

## Key Areas of Focus

- **Technology and Innovation**: Apple is renowned for its revolutionary products, including Macs, iPads, iPhones, Apple Watches, and more. Each product is designed with a focus on simplicity, elegance, and user experience.

- **Entertainment and Media**: Apple offers a vast array of entertainment options through services like Apple TV+, Apple Music, and Apple Podcasts. These services provide access to a wide range of movies, TV shows, music, and podcasts, catering to a diverse audience.

- **Sustainability**: Apple is committed to reducing its environmental impact. The company has set ambitious goals for carbon neutrality and is working to source 100% renewable energy for its operations. Apple also encourages sustainable practices among its suppliers.

- **Community and Culture**: Apple fosters a culture of creativity and collaboration. From its headquarters in Cupertino to retail stores worldwide, Apple stores serve as hubs for innovation, learning, and community engagement.

## Leadership and Careers

Apple's leadership is known for driving innovation and excellence. The company offers a wide range of career opportunities across various departments, including engineering, marketing, design, and more. Apple values diversity, inclusion, and collaboration, providing a supportive environment for all employees.

## Corporate Social Responsibility

Apple is dedicated to making a positive impact on society and the environment. The company is involved in numerous initiatives aimed at promoting education, digital literacy, and environmental sustainability. Apple also supports various charitable organizations and community programs.

## Contact and Legal Information

For any inquiries or support, you can reach out to Apple through their contact page. Apple also provides detailed legal and ethical guidelines, ensuring transparency and compliance in all their operations.

## Join the Apple Community

Whether you're a tech enthusiast, a business owner, or simply interested in staying ahead of the curve, Apple offers something for everyone. Join the Apple community and experience innovation, design, and technology like never before.

---

**Apple Values**: Integrity, Transparency, Innovation, Design, and People.

**Apple Mission**: To make the world a better place through technology and creativity.

---

Apple continues to lead the tech industry with its commitment to innovation, sustainability, and community. Explore Apple's products, services, and values to see how they can enhance your daily life and inspire your future endeavors. 

---

*Visit [Apple’s Website](https://www.apple.com) to learn more.*

In [7]:
def stream_brochure(company_name, url):
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    system_prompt = """You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information."""
    
    stream = ollama.chat(
        model="qwen2.5:7b",  # or whatever local model you have pulled
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": create_brochure(company_name, url)}
        ],
        stream=True
    )
    
    for chunk in stream:
        text = chunk['message']['content']
        response += text
        update_display(Markdown(response), display_id=display_handle.display_id)

In [8]:
stream_brochure("apple","https://www.apple.com")

---
# Discover Apple: Where Innovation Meets Heart

## About Apple

Apple, founded in 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne, has grown from a garage-based startup into a global icon of innovation and quality. Today, under the leadership of Tim Cook, the company continues to push the boundaries of technology and design. Apple is committed to creating products that are not only revolutionary but also user-friendly and accessible, ensuring that every customer can trust their personal data and benefit from Apple's commitment to privacy and security.

### Core Values

At Apple, innovation and creativity are not just business goals but core values. The company is dedicated to creating products that are not only groundbreaking but also designed with the user in mind. Apple’s commitment to privacy and security is another hallmark of its ethos, ensuring that customers can trust their personal data. This is reflected in its focus on data protection, secure transactions, and ethical use of technology.

### Corporate Responsibility

Apple is committed to making a positive impact on the planet and society. The company has set ambitious goals to be carbon neutral across its entire business, including supply chain and product life cycles by 2030. Apple’s commitment to sustainability is evident in its diverse product line, which includes Mac computers, iPads, iPhones, Apple Watches, and the latest in home and wearable technology. The company also focuses on reducing its carbon footprint, using 100% renewable energy for all its operational facilities worldwide, and investing in environmental and social initiatives.

## Leadership and Careers

Apple’s leadership team is known for its dedication to excellence and innovation. The company offers a range of career opportunities across various departments, from engineering and design to marketing and finance. Apple’s culture of innovation and collaboration provides employees with a dynamic and rewarding work environment.

### Career Opportunities

Explore job openings and opportunities at Apple. Whether you're a recent graduate or an experienced professional, there are roles available that align with your skills and interests. From software development to retail operations, Apple offers a diverse range of career paths. Apple’s employee benefits and development programs ensure that employees are not only supported but also empowered to reach their full potential.

## Investors and Sustainability

Apple is a public company listed on the NASDAQ stock exchange, and its financial performance is closely watched by investors. The company’s strong financial health, steady growth, and commitment to sustainability make it a leading investment choice. Apple’s sustainability reports provide detailed insights into its environmental and social impact, offering transparency and accountability to shareholders.

### Sustainability Reports

Apple’s sustainability reports outline its progress in reducing carbon emissions, conserving resources, and promoting social responsibility. These reports are available on the company’s investor relations page, providing a comprehensive view of Apple’s commitment to ethical business practices.

## Customers

Apple’s products are used by millions of customers worldwide. From students and professionals to families and creatives, Apple products have become essential tools in various fields. The company’s commitment to user experience and design has made its products beloved by a diverse range of users.

## Contact and Legal

For more information, you can contact Apple through their official website or customer support channels. Apple’s legal and ethical guidelines ensure that its operations comply with local and international laws, protecting its employees, customers, and the environment.

### Contact Information

- **Website:** <https://www.apple.com/>
- **Customer Support:** <https://www.apple.com/support/>

### Legal and Ethical Guidelines

Apple adheres to strict ethical and legal standards. The company’s legal and compliance team ensures that all business practices comply with relevant laws and regulations, maintaining the highest standards of integrity and responsibility.

## Conclusion

Apple is more than just a technology company; it’s a symbol of creativity, innovation, and a commitment to making the world a better place. With a strong focus on sustainability and ethical business practices, Apple continues to inspire and lead the way in the tech industry.

Join the Apple community and be part of a global movement dedicated to excellence and innovation. Explore careers, learn about the company’s values, and stay informed about the latest developments in the world of technology.

---

This brochure highlights Apple's core values, career opportunities, commitment to sustainability, and the impact on customers, making it a compelling introduction to the company for prospective customers, investors, and recruits.